In [0]:
import pyspark.sql.functions as F

In [0]:
def validate_flag_threshold(df, flag_column: str, tolerance: float):
    """
    Validates percentage of FALSE values for a given data quality flag.

    Args:
        df: dataframe to validate
        flag_column: boolean flag column name
        tolerance: max allowed percentage (0–1)

    Raises:
        Exception if threshold exceeded
    """

    metrics = (df.select(
            F.count("*").alias("total"),
            F.sum(F.when(F.col(flag_column) == False, 1).otherwise(0))
            .alias("invalid")
        )
        .collect()[0]
    )

    total = metrics["total"]
    invalid = metrics["invalid"]

    ratio = invalid / total if total > 0 else 0

    print(f"[DQ CHECK] - Do NOT {flag_column}: {invalid}/{total} ({ratio:.2%})")

    if ratio > tolerance:
        raise Exception(
            f"DATA ALERT → {flag_column} exceeded tolerance "
            f"({ratio:.2%} > {tolerance:.2%})"
        )

        

In [0]:
def validate_non_empty(df, msg: str):
    """
    Validate if the dataframe contains data
    """
    if df.count() == 0:
        raise Exception(msg)